# Notebook 06 — Graphe Social Instagram

Construit le graphe de relations à partir des DMs Instagram.
Pondération par volume de messages + multiplicateur close friends.

Pas de Spark : les données sont des JSON individuels par conversation, lus séquentiellement. Le volume (~100 nœuds max) ne justifie pas Spark.

Input  : data/raw/INSTAGRAM/messages/inbox/

Output : data/warehouse/social_graph/ (dossier parquet)

In [1]:
import json
import os
import re
import sys

import pandas as pd

# ── Config centrale ────────────────────────────────────────────────────────────
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import WAREHOUSE, RAW_DATA, CLOSE_FRIENDS, CLOSE_FRIENDS_MULTIPLIER, MIN_MESSAGES


In [ ]:
# ── CHEMINS ───────────────────────────────────────────────────────────────────
INBOX = os.path.join(RAW_DATA, 'INSTAGRAM', 'your_instagram_activity', 'messages', 'inbox')
os.makedirs(WAREHOUSE, exist_ok=True)

print('Warehouse:', WAREHOUSE)
print('Inbox conversations:', len(os.listdir(INBOX)) if os.path.exists(INBOX) else 'N/A')

In [ ]:
# ── CLOSE FRIENDS ─────────────────────────────────────────────────────────────
# Chargés depuis config.py — modifier config.py à la racine du projet.
# CLOSE_FRIENDS_MULTIPLIER booste le poids de ces relations pour refléter
# la qualité du lien, pas uniquement le volume brut.
print(f"Close friends configurés : {len(CLOSE_FRIENDS)}")
print(f"Multiplicateur           : ×{CLOSE_FRIENDS_MULTIPLIER}")
print(f"Seuil minimum messages   : {MIN_MESSAGES}")

In [ ]:
# ── PARSING DE L'INBOX ────────────────────────────────────────────────────────
def _parse_conversation(conv_dir: str, folder: str) -> dict | None:
    # Trier les fichiers message_*.json par numéro (Instagram pagine à 10 000 msgs/fichier)
    msg_files = sorted(
        [f for f in os.listdir(conv_dir) if f.startswith("message_") and f.endswith(".json")],
        key=lambda f: int(re.search(r'(\d+)', f).group(1)),
        reverse=True,
    )
    if not msg_files:
        return None

    # Premier fichier pour les métadonnées (participants)
    with open(os.path.join(conv_dir, msg_files[0]), encoding="utf-8") as f:
        data = json.load(f)

    # Garder uniquement les conversations 1-to-1 (pas les groupes)
    if len(data.get("participants", [])) != 2:
        return None

    # Sommer les messages sur TOUS les fichiers de la conversation
    msg_count = 0
    for fname in msg_files:
        with open(os.path.join(conv_dir, fname), encoding="utf-8") as f:
            msg_count += len(json.load(f).get("messages", []))

    # MIN_MESSAGES : seuil pour exclure bots, contacts one-shot et conversations ponctuelles
    if msg_count < MIN_MESSAGES:
        return None

    # Extraire le prénom depuis le format Instagram : "prenom_1234567890"
    label = re.split(r'_\d{10,}', folder)[0].lower()
    node_id = folder.lower()
    is_close = (label in CLOSE_FRIENDS) or (node_id in CLOSE_FRIENDS)

    return {"node_id": node_id, "label": label, "message_count": msg_count, "in_close_friends": is_close}


records = []
for folder in os.listdir(INBOX):
    conv_dir = os.path.join(INBOX, folder)
    if not os.path.isdir(conv_dir):
        continue
    result = _parse_conversation(conv_dir, folder)
    if result:
        records.append(result)

df = pd.DataFrame(records)

# Pondération : close friends × CLOSE_FRIENDS_MULTIPLIER (défaut 2.0)
# Reflète la qualité de la relation, pas uniquement le volume brut
df["weight"] = df.apply(
    lambda row: row["message_count"] * CLOSE_FRIENDS_MULTIPLIER if row["in_close_friends"] else float(row["message_count"]),
    axis=1,
)
df = df.sort_values("weight", ascending=False).reset_index(drop=True)

print(f"Conversations retenues (>= {MIN_MESSAGES} msgs) : {len(df)}")
print(f"Close friends : {df['in_close_friends'].sum()}")
print()
print(df[["label", "message_count", "in_close_friends", "weight"]].head(30).to_string())

In [ ]:
# ── SAUVEGARDE ────────────────────────────────────────────────────────────────
# Format dossier parquet (cohérent avec les autres tables warehouse)
# Le dashboard lit depuis ce dossier via pd.read_parquet(directory).

out_dir = os.path.join(WAREHOUSE, "social_graph")
os.makedirs(out_dir, exist_ok=True)
df.to_parquet(os.path.join(out_dir, "part-0.parquet"), index=False)

print(f"Sauvegardé : {out_dir}")
print(f"  {len(df)} nœuds, {df['message_count'].sum():,} messages total")

# Vérification
check = pd.read_parquet(out_dir)
print(f"  Vérification lecture dossier : {len(check)} lignes OK")